In [16]:

import sys
import platform
import subprocess

print("=== INFORMASI LINGKUNGAN ===")
print("Versi Python:", sys.version.split()[0])
print("Sistem operasi:", platform.system())
print("Arsitektur:", platform.machine())

print("\n=== VERSI JAVA ===")
hasil = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(hasil.stderr if hasil.stderr else hasil.stdout)


=== INFORMASI LINGKUNGAN ===
Versi Python: 3.13.16
Sistem operasi: Linux
Arsitektur: x86_64

=== VERSI JAVA ===
openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)



In [18]:

!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless
%pip install -q pyspark==4.0.1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [19]:

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])


JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [21]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EksplorasiMahadataKesehatan")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Versi Apache Spark:", spark.version)
print("Mode eksekusi:", spark.sparkContext.master)
print("Spark berhasil dijalankan!")


Versi Apache Spark: 4.0.1
Mode eksekusi: local[2]
Spark berhasil dijalankan!


In [22]:

data_uji = [
    (1, "Normal"),
    (2, "Berisiko"),
    (3, "Normal")
]

df_uji = spark.createDataFrame(
    data_uji,
    ["id", "status"]
)

df_uji.show()

print("Jumlah baris:", df_uji.count())


+---+--------+
| id|  status|
+---+--------+
|  1|  Normal|
|  2|Berisiko|
|  3|  Normal|
+---+--------+

Jumlah baris: 3


In [23]:

import os
import urllib.request
import hashlib

url = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/heart-disease/"
    "processed.cleveland.data"
)

os.makedirs("data", exist_ok=True)
file_path = "data/processed.cleveland.data"

urllib.request.urlretrieve(url, file_path)

with open(file_path, "rb") as f:
    checksum = hashlib.sha256(f.read()).hexdigest()

print("Dataset berhasil diunduh!")
print("Lokasi:", file_path)
print("Ukuran file:", os.path.getsize(file_path), "bytes")
print("SHA-256:", checksum)


Dataset berhasil diunduh!
Lokasi: data/processed.cleveland.data
Ukuran file: 18461 bytes
SHA-256: a74b7efa387bc9d108d7d0115d831fe9b414b29ae7124f331b622b4efa0427c8


In [24]:

nama_kolom = [
    "age", "sex", "cp", "trestbps",
    "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope",
    "ca", "thal", "num"
]

df = spark.read.csv(
    file_path,
    header=False,
    inferSchema=False
)

df = df.toDF(*nama_kolom)

print("=== STRUKTUR DATASET ===")
df.printSchema()

print("\n=== LIMA BARIS PERTAMA ===")
df.show(5, truncate=False)

print("Jumlah baris:", df.count())
print("Jumlah kolom:", len(df.columns))


=== STRUKTUR DATASET ===
root
 |-- age: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- cp: string (nullable = true)
 |-- trestbps: string (nullable = true)
 |-- chol: string (nullable = true)
 |-- fbs: string (nullable = true)
 |-- restecg: string (nullable = true)
 |-- thalach: string (nullable = true)
 |-- exang: string (nullable = true)
 |-- oldpeak: string (nullable = true)
 |-- slope: string (nullable = true)
 |-- ca: string (nullable = true)
 |-- thal: string (nullable = true)
 |-- num: string (nullable = true)


=== LIMA BARIS PERTAMA ===
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+---+
|age |sex|cp |trestbps|chol |fbs|restecg|thalach|exang|oldpeak|slope|ca |thal|num|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+---+
|63.0|1.0|1.0|145.0   |233.0|1.0|2.0    |150.0  |0.0  |2.3    |3.0  |0.0|6.0 |0  |
|67.0|1.0|4.0|160.0   |286.0|0.0|2.0    |108.0  |1.0  |1.5    |2.0  |3.0|3.0 |2  |
|67.0|1.0|4.0|1

In [25]:

from pyspark.sql import functions as F

print("=== PEMERIKSAAN MISSING VALUES ===")

# Menghitung nilai kosong, tanda ?, dan NULL
missing_expr = [
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == "?") |
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]

hasil_missing = df.agg(*missing_expr).first().asDict()

for kolom, jumlah in hasil_missing.items():
    print(f"{kolom}: {jumlah} missing values")

print("\nTotal missing values:", sum(hasil_missing.values()))


=== PEMERIKSAAN MISSING VALUES ===
age: 0 missing values
sex: 0 missing values
cp: 0 missing values
trestbps: 0 missing values
chol: 0 missing values
fbs: 0 missing values
restecg: 0 missing values
thalach: 0 missing values
exang: 0 missing values
oldpeak: 0 missing values
slope: 0 missing values
ca: 4 missing values
thal: 2 missing values
num: 0 missing values

Total missing values: 6


In [26]:

print("=== DISTRIBUSI TARGET PENYAKIT JANTUNG ===")

df.groupBy("num") \
  .count() \
  .orderBy(F.col("num").cast("int")) \
  .show()

print("=== PEMERIKSAAN DATA DUPLIKAT ===")

total_baris = df.count()
baris_unik = df.distinct().count()

print("Total baris:", total_baris)
print("Baris unik:", baris_unik)
print("Baris duplikat:", total_baris - baris_unik)


=== DISTRIBUSI TARGET PENYAKIT JANTUNG ===
+---+-----+
|num|count|
+---+-----+
|  0|  164|
|  1|   55|
|  2|   36|
|  3|   35|
|  4|   13|
+---+-----+

=== PEMERIKSAAN DATA DUPLIKAT ===
Total baris: 303
Baris unik: 303
Baris duplikat: 0


In [27]:

from pyspark.sql import functions as F

# Memastikan nama kolom sudah benar
print("Nama kolom:", df.columns)

# Kondisi awal untuk mendeteksi missing values
kondisi_missing = F.lit(False)

for kolom in df.columns:
    kondisi_missing = (
        kondisi_missing |
        F.col(kolom).isNull() |
        (F.trim(F.col(kolom)) == "?") |
        (F.trim(F.col(kolom)) == "")
    )

# Mengambil baris yang memiliki nilai hilang
df_missing = df.filter(kondisi_missing)

print("\n=== PEMERIKSAAN BARIS BERMASALAH ===")
print("Jumlah data awal:", df.count())
print("Baris dengan missing values:", df_missing.count())

print("\n=== DETAIL BARIS BERMASALAH ===")
df_missing.select(
    "age", "sex", "ca", "thal", "num"
).show(truncate=False)


Nama kolom: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

=== PEMERIKSAAN BARIS BERMASALAH ===
Jumlah data awal: 303
Baris dengan missing values: 6

=== DETAIL BARIS BERMASALAH ===
+----+---+---+----+---+
|age |sex|ca |thal|num|
+----+---+---+----+---+
|53.0|0.0|0.0|?   |0  |
|52.0|1.0|?  |3.0 |0  |
|43.0|1.0|?  |7.0 |1  |
|52.0|1.0|0.0|?   |2  |
|58.0|1.0|?  |7.0 |0  |
|38.0|1.0|?  |3.0 |0  |
+----+---+---+----+---+



In [28]:

from pyspark.sql import functions as F

# 1. Menghapus baris yang memiliki missing values
df_bersih = df.filter(~kondisi_missing)

# 2. Mengubah seluruh kolom menjadi tipe numerik
for kolom in df_bersih.columns:
    df_bersih = df_bersih.withColumn(
        kolom,
        F.col(kolom).cast("double")
    )

# 3. Membuat label klasifikasi biner
df_bersih = df_bersih.withColumn(
    "label",
    F.when(F.col("num") == 0, 0.0)
     .otherwise(1.0)
)

# 4. Memeriksa hasil pembersihan
print("=== HASIL PEMBERSIHAN DATA ===")
print("Data awal:", df.count())
print("Data bersih:", df_bersih.count())
print("Data dihapus:", df.count() - df_bersih.count())

# 5. Memeriksa distribusi label
print("\n=== DISTRIBUSI LABEL ===")
df_bersih.groupBy("label") \
    .count() \
    .orderBy("label") \
    .show()

# 6. Memeriksa apakah masih ada NULL
print("=== PEMERIKSAAN NULL SETELAH KONVERSI ===")

df_bersih.select([
    F.sum(
        F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1)
         .otherwise(0)
    ).alias(c)
    for c in df_bersih.columns
]).show()

# 7. Menampilkan struktur data
print("=== STRUKTUR DATA BERSIH ===")
df_bersih.printSchema()


=== HASIL PEMBERSIHAN DATA ===
Data awal: 303
Data bersih: 297
Data dihapus: 6

=== DISTRIBUSI LABEL ===
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  160|
|  1.0|  137|
+-----+-----+

=== PEMERIKSAAN NULL SETELAH KONVERSI ===
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+---+-----+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|num|label|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+---+-----+
|  0|  0|  0|       0|   0|  0|      0|      0|    0|      0|    0|  0|   0|  0|    0|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+---+-----+

=== STRUKTUR DATA BERSIH ===
root
 |-- age: double (nullable = true)
 |-- sex: double (nullable = true)
 |-- cp: double (nullable = true)
 |-- trestbps: double (nullable = true)
 |-- chol: double (nullable = true)
 |-- fbs: double (nullable = true)
 |-- restecg: double (nullable = true)
 |-- thalach: double (nullable = true)
 |-- 

In [29]:

# Menentukan 13 atribut yang digunakan sebagai fitur
fitur = [
    "age", "sex", "cp", "trestbps",
    "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak",
    "slope", "ca", "thal"
]

# Memisahkan fitur dan label
# Kolom num tidak dimasukkan agar tidak terjadi data leakage
df_model = df_bersih.select(*fitur, "label")

print("=== PERSIAPAN FITUR ===")
print("Jumlah fitur:", len(fitur))
print("Nama fitur:", fitur)
print("Jumlah data:", df_model.count())

# Pembagian data 70% latih dan 30% uji
df_latih, df_uji = df_model.randomSplit(
    [0.7, 0.3],
    seed=42
)

# Menyimpan hasil pembagian selama runtime berjalan
df_latih = df_latih.cache()
df_uji = df_uji.cache()

jumlah_latih = df_latih.count()
jumlah_uji = df_uji.count()

print("\n=== PEMBAGIAN DATA ===")
print("Data latih:", jumlah_latih)
print("Data uji:", jumlah_uji)
print("Total:", jumlah_latih + jumlah_uji)

print("\n=== DISTRIBUSI LABEL DATA LATIH ===")
df_latih.groupBy("label").count().orderBy("label").show()

print("=== DISTRIBUSI LABEL DATA UJI ===")
df_uji.groupBy("label").count().orderBy("label").show()



=== PERSIAPAN FITUR ===
Jumlah fitur: 13
Nama fitur: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Jumlah data: 297

=== PEMBAGIAN DATA ===
Data latih: 220
Data uji: 77
Total: 297

=== DISTRIBUSI LABEL DATA LATIH ===
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  112|
|  1.0|  108|
+-----+-----+

=== DISTRIBUSI LABEL DATA UJI ===
+-----+-----+
|label|count|
+-----+-----+
|  0.0|   48|
|  1.0|   29|
+-----+-----+



In [30]:

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorIndexer
from pyspark.ml.classification import DecisionTreeClassifier
import time

# 1. Menggabungkan 13 fitur menjadi vektor
assembler = VectorAssembler(
    inputCols=fitur,
    outputCol="fitur_vektor",
    handleInvalid="error"
)

# 2. Mengidentifikasi fitur kategorikal
indexer = VectorIndexer(
    inputCol="fitur_vektor",
    outputCol="features",
    maxCategories=5,
    handleInvalid="keep"
)

# 3. Menentukan algoritma Decision Tree
decision_tree = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    maxBins=32,
    seed=42
)

# 4. Membuat pipeline pemodelan
pipeline = Pipeline(stages=[
    assembler,
    indexer,
    decision_tree
])

# 5. Melatih model dan mengukur waktu
mulai = time.perf_counter()

model = pipeline.fit(df_latih)

waktu_latih = time.perf_counter() - mulai

# 6. Melakukan prediksi pada data uji
hasil_prediksi = model.transform(df_uji)

# 7. Mengambil informasi model
model_pohon = model.stages[-1]

print("=== HASIL PELATIHAN DECISION TREE ===")
print("Algoritma: Decision Tree Classifier")
print("Jumlah data latih:", jumlah_latih)
print("Jumlah data uji:", jumlah_uji)
print("Kedalaman pohon:", model_pohon.depth)
print("Jumlah node:", model_pohon.numNodes)
print("Waktu pelatihan:", round(waktu_latih, 4), "detik")

print("\n=== CONTOH HASIL PREDIKSI ===")
hasil_prediksi.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)


=== HASIL PELATIHAN DECISION TREE ===
Algoritma: Decision Tree Classifier
Jumlah data latih: 220
Jumlah data uji: 77
Kedalaman pohon: 5
Jumlah node: 39
Waktu pelatihan: 3.9546 detik

=== CONTOH HASIL PREDIKSI ===
+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0.0  |0.0       |[0.9629629629629629,0.037037037037037035]|
|1.0  |1.0       |[0.2,0.8]                                |
|0.0  |0.0       |[1.0,0.0]                                |
|1.0  |0.0       |[0.8,0.2]                                |
|1.0  |1.0       |[0.0,1.0]                                |
|0.0  |0.0       |[0.8,0.2]                                |
|1.0  |1.0       |[0.0,1.0]                                |
|0.0  |0.0       |[0.9629629629629629,0.037037037037037035]|
|0.0  |0.0       |[0.9629629629629629,0.037037037037037035]|
|0.0  |0.0       |[0.9629629629629629,0.037037037037037

In [31]:

# 1. Menghitung hasil prediksi seluruh data uji
hasil_evaluasi = (
    hasil_prediksi
    .groupBy("label", "prediction")
    .count()
    .collect()
)

# 2. Membentuk confusion matrix
jumlah = {
    (float(r["label"]), float(r["prediction"])): r["count"]
    for r in hasil_evaluasi
}

TN = jumlah.get((0.0, 0.0), 0)
FP = jumlah.get((0.0, 1.0), 0)
FN = jumlah.get((1.0, 0.0), 0)
TP = jumlah.get((1.0, 1.0), 0)

total = TP + TN + FP + FN
assert total == jumlah_uji, "Jumlah prediksi tidak sesuai data uji"

print("=== CONFUSION MATRIX ===")
print("                    Prediksi 0   Prediksi 1")
print(f"Label aktual 0       {TN:>5}        {FP:>5}")
print(f"Label aktual 1       {FN:>5}        {TP:>5}")

# 3. Menghitung metrik evaluasi untuk kelas 1
accuracy = (TP + TN) / total if total else 0
precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1_score = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

# 4. Menampilkan hasil evaluasi
print("\n=== HASIL EVALUASI MODEL ===")
print("Total data uji:", total)
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score : {f1_score:.4f} ({f1_score*100:.2f}%)")

print("\n=== RINCIAN PREDIKSI ===")
print("True Positive :", TP)
print("True Negative :", TN)
print("False Positive:", FP)
print("False Negative:", FN)


=== CONFUSION MATRIX ===
                    Prediksi 0   Prediksi 1
Label aktual 0          42            6
Label aktual 1           7           22

=== HASIL EVALUASI MODEL ===
Total data uji: 77
Accuracy : 0.8312 (83.12%)
Precision: 0.7857 (78.57%)
Recall   : 0.7586 (75.86%)
F1-score : 0.7719 (77.19%)

=== RINCIAN PREDIKSI ===
True Positive : 22
True Negative : 42
False Positive: 6
False Negative: 7


In [32]:

import os
import sys
import json
import csv
import hashlib
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version

os.makedirs("hasil", exist_ok=True)

# 1. Mengambil versi lingkungan
java_info = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

# 2. Menghitung ulang checksum dataset
with open(file_path, "rb") as f:
    hash_dataset = hashlib.sha256(f.read()).hexdigest()

# 3. Membuat ringkasan eksperimen
ringkasan = {
    "waktu_pencatatan_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "lingkungan": {
        "python": sys.version.split()[0],
        "pyspark": version("pyspark"),
        "spark": spark.version,
        "java": java_info.stderr.splitlines()[0],
        "mode": spark.sparkContext.master
    },
    "dataset": {
        "sumber": "UCI Cleveland Heart Disease",
        "sha256": hash_dataset,
        "data_awal": df.count(),
        "data_bersih": df_bersih.count(),
        "data_latih": jumlah_latih,
        "data_uji": jumlah_uji
    },
    "model": {
        "algoritma": "Decision Tree",
        "max_depth": decision_tree.getMaxDepth(),
        "max_bins": decision_tree.getMaxBins(),
        "max_categories": indexer.getMaxCategories(),
        "seed": 42,
        "kedalaman_aktual": model_pohon.depth,
        "jumlah_node": model_pohon.numNodes,
        "waktu_latih_detik": waktu_latih
    },
    "evaluasi": {
        "accuracy": accuracy,
        "precision_kelas_1": precision,
        "recall_kelas_1": recall,
        "f1_kelas_1": f1_score,
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN)
    }
}

# 4. Menyimpan ringkasan JSON
with open("hasil/hasil_eksperimen.json", "w") as f:
    json.dump(ringkasan, f, indent=4)

# 5. Menyimpan prediksi seluruh data uji
baris_prediksi = hasil_prediksi.select(
    "label", "prediction", "probability"
).collect()

with open("hasil/prediksi_data_uji.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "label", "prediction", "probabilitas_kelas_1"
    ])

    for baris in baris_prediksi:
        writer.writerow([
            baris["label"],
            baris["prediction"],
            float(baris["probability"][1])
        ])

print("=== FILE BERHASIL DISIMPAN ===")
print(os.listdir("hasil"))
print("Jumlah prediksi tersimpan:", len(baris_prediksi))
print("Versi PySpark:", version("pyspark"))
print("Waktu pelatihan:", round(waktu_latih, 4), "detik")


=== FILE BERHASIL DISIMPAN ===
['prediksi_data_uji.csv', 'hasil_eksperimen.json']
Jumlah prediksi tersimpan: 77
Versi PySpark: 4.0.1
Waktu pelatihan: 3.9546 detik


In [33]:

import os
import zipfile

nama_zip = "hasil_eksperimen_mahadata.zip"

with zipfile.ZipFile(
    nama_zip, "w", zipfile.ZIP_DEFLATED
) as zipf:
    zipf.write(
        "hasil/hasil_eksperimen.json"
    )
    zipf.write(
        "hasil/prediksi_data_uji.csv"
    )

print("=== ARSIP HASIL EKSPERIMEN ===")
print("Nama file:", nama_zip)
print("Ukuran:", os.path.getsize(nama_zip), "bytes")

with zipfile.ZipFile(nama_zip, "r") as zipf:
    print("Isi arsip:", zipf.namelist())


=== ARSIP HASIL EKSPERIMEN ===
Nama file: hasil_eksperimen_mahadata.zip
Ukuran: 973 bytes
Isi arsip: ['hasil/hasil_eksperimen.json', 'hasil/prediksi_data_uji.csv']


In [34]:

from google.colab import files

files.download("hasil_eksperimen_mahadata.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>